In [ ]:
#GET LINKS

import asyncio
import csv
import json
from playwright.async_api import async_playwright, Playwright
import numpy as np

meal_type = ['dinner-recipes', 'breakfast', 'appetizers', 'side-dishes', 'lunch']

meal_lists = {meal: [] for meal in meal_type}
seen_link = set()

async def max(input):

    page_count = []
    for i in input:
        try:
            c = int(i)
            page_count.append(c)
        except ValueError:
            pass
        
    return np.max(page_count)


async def info(playwright: Playwright):

    chromium = playwright.chromium
    browser = await chromium.launch()
    page = await browser.new_page()

    for meal in meal_type:

        if meal == 'lunch':
            
            await page.goto(f"https://cooking.nytimes.com/68861692-nyt-cooking/20404912-lunch-ideas")

            pages = page.locator(f'a[href*="/20404912-lunch-ideas?page="]')
            pagecount = await pages.all_text_contents()
            pagecount = await max(pagecount)

            for i in range(int(pagecount)):

                await page.goto(f'https://cooking.nytimes.com/68861692-nyt-cooking/20404912-lunch-ideas?page={i + 1}')
                recipe_links = page.locator('a[href*="/recipes/"]')

                for j in range(await recipe_links.count()):
                    link = await recipe_links.nth(j).get_attribute('href')

                    if 'nytimes.com' in link:
                        link = link
                    else:
                        link = 'https://cooking.nytimes.com' + link

                    if link and link not in seen_link:
                        meal_lists['lunch'].append(link)
                        seen_link.add(link)

        else:

            await page.goto(f"https://cooking.nytimes.com/topics/{meal}")

            pages = page.locator(f'a[href*="/{meal}?page="]')
            pagecount = await pages.all_text_contents()
            pagecount = await max(pagecount)


            for i in range(int(pagecount)):
                await page.goto(f'https://cooking.nytimes.com/topics/{meal}?page={i + 1}')
                recipe_links = page.locator('a[href*="/recipes/"]')

                for j in range(await recipe_links.count()):
                    link = await recipe_links.nth(j).get_attribute('href')

                    if 'nytimes.com' in link:
                        link = link
                    else:
                        link = 'https://cooking.nytimes.com' + link

                    if link and link not in seen_link:
                        meal_lists[meal].append(link)
                        seen_link.add(link)

In [30]:
async def main():
    async with async_playwright() as playwright:
        return await info(playwright)

await main()


In [31]:
with open('nyt_recipe_dict.json', 'w') as file:
    json.dump(meal_lists, file, indent = 2)

with open('nyt_recipe_links.csv', 'w') as file:
    writer = csv.writer(file)

    for link in seen_link:
        writer.writerow([link])

In [32]:
print("total breakfast links:", len(meal_lists["breakfast"]))
print("unique breakfast links:", len(set(meal_lists["breakfast"])))

total breakfast links: 945
unique breakfast links: 945
